# 🔥 **SISTEM MONITORING & PERINGATAN DINI KEBAKARAN HUTAN (KARHUTLA)**
### **Tim Pengembang : Testarossa's Vanguard — Universitas Ahmad Dahlan**
---
Notebook ini meng-host **Python MQTT Backend & REST API Server** yang terhubung dua arah (*bidirectional*) dengan simulasi Wokwi ESP32:
1. **Simulasi Wokwi**: ESP32 membaca sensor suhu & kelembapan udara (DHT22), kepekatan asap/gas (MQ-2 AO), dan kelembapan tanah gambut (Potensiometer). Mengendalikan 3 LED (Hijau, Kuning, Merah) dan Buzzer sirene.
2. **MQTT Broker**: `broker.hivemq.com:1883` (Topic Telemetri: `forest/fire/telemetry`, Topic Kontrol: `forest/fire/control`).
3. **Python Decision Engine**: Melakukan evaluasi risiko multivariat (*Normal*, *Warning*, *Danger*) serta menghitung Indeks Risiko Kebakaran gabungan (Temp 35%, Soil 25%, Smoke 40%).
4. **Kontrol Balik (Feedback)**: Server Colab mengirimkan status risiko ke ESP32 agar aktuator fisik di Wokwi merespons secara otomatis.
5. **Frontend Bento UI**: Menyajikan dashboard modern berkinerja tinggi (`frontend/dashboard.html`) langsung di dalam tampilan Google Colab.

### **1 — Install Dependensi Google Colab**
Instalasi library `paho-mqtt`, `fastapi`, `uvicorn`, `nest-asyncio`, dan `requests`.

In [ ]:
%pip install -q paho-mqtt fastapi uvicorn nest-asyncio requests
print("✅ Seluruh dependensi berhasil diinstal!")


### **2 — Ambil File Frontend Dashboard**
Mengambil file `frontend/dashboard.html` dari repository GitHub publik `firzal098/Peringatan-Dini-Kebakaran` jika belum ada di workspace.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/firzal098/Peringatan-Dini-Kebakaran.git"
REPO_DIR = "Peringatan-Dini-Kebakaran"

# Cek apakah file dashboard.html sudah ada di direktori kerja saat ini
if os.path.exists("frontend/dashboard.html"):
    DASHBOARD_FILE = "frontend/dashboard.html"
elif os.path.exists(os.path.join(REPO_DIR, "frontend", "dashboard.html")):
    DASHBOARD_FILE = os.path.join(REPO_DIR, "frontend", "dashboard.html")
else:
    print("📥 Mengkloning repository Vanguard dari GitHub...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    DASHBOARD_FILE = os.path.join(REPO_DIR, "frontend", "dashboard.html")

print(f"✅ Dashboard HTML siap di: {os.path.abspath(DASHBOARD_FILE)}")


### **3 — Konfigurasi MQTT & Decision Engine**
Menyiapkan client MQTT untuk berlangganan topik telemetri Wokwi dan logika penentuan status risiko kebakaran.

In [ ]:
import time
import json
import threading
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import deque
import paho.mqtt.client as mqtt

# ── KONFIGURASI MQTT & PORT ───────────────────────────────────
MQTT_BROKER     = "broker.hivemq.com"
MQTT_PORT       = 1883
MQTT_CLIENT_ID  = f"colab_karhutla_backend_{int(time.time())}"
TOPIC_TELEMETRY = "testarossas_vanguard/fire/karhutla/telemetry"
TOPIC_CONTROL   = "testarossas_vanguard/fire/karhutla/control"

HTTP_PORT = 8000
WIB = ZoneInfo("Asia/Jakarta")

# ── STORAGE STATE (THREAD-SAFE) ───────────────────────────────
data_lock = threading.Lock()
MAX_HISTORY = 100
telemetry_history = deque(maxlen=MAX_HISTORY)

latest_state = {
    "device_id": "ESP32_FIRE_NODE_01",
    "temperature": 27.0,
    "humidity": 65.0,
    "soil_moisture": 52.0,
    "smoke_ppm": 90,
    "status": "Normal",
    "risk_index": 18,
    "mqtt_connected": False,
    "messages_received": 0,
    "last_telemetry_time": None,
    "control_override": None  # "AUTO" atau "Normal"/"Warning"/"Danger"
}

# ── MULTIVARIATE RISK EVALUATION ─────────────────────────────
def evaluate_risk(temp, hum, soil, smoke):
    danger_combined = (temp >= 40.0 and soil <= 20.0 and smoke >= 400.0)
    danger_smoke    = (smoke >= 600.0)
    danger_heat     = (temp >= 45.0 and smoke >= 250.0)
    
    if danger_combined or danger_smoke or danger_heat:
        return "Danger"
    if temp >= 32.0 or soil <= 35.0 or smoke >= 200.0 or hum < 45.0:
        return "Warning"
    return "Normal"

def calculate_risk_percentage(temp, hum, soil, smoke):
    temp_score = min(100, max(0, ((temp - 20) / 30) * 100))
    soil_dry_score = min(100, max(0, 100 - soil))
    smoke_score = min(100, max(0, (smoke / 350) * 100))
    risk = round((temp_score * 0.35) + (soil_dry_score * 0.25) + (smoke_score * 0.40))
    return min(100, max(5, risk))

# ── MQTT CLIENT HANDLERS ─────────────────────────────────────
mqtt_client = mqtt.Client(
    callback_api_version=mqtt.CallbackAPIVersion.VERSION2,
    client_id=MQTT_CLIENT_ID
)

def on_connect(client, userdata, flags, reason_code, properties):
    if reason_code == 0:
        with data_lock:
            latest_state["mqtt_connected"] = True
        print(f"✅ [MQTT] Terhubung ke {MQTT_BROKER}:{MQTT_PORT}")
        client.subscribe(TOPIC_TELEMETRY)
        print(f"📡 [MQTT] Berlangganan topik: '{TOPIC_TELEMETRY}'")
    else:
        print(f"❌ [MQTT] Koneksi gagal (rc={reason_code})")

def on_message(client, userdata, message):
    try:
        payload_raw = message.payload.decode("utf-8").strip()
        data = json.loads(payload_raw)
        now = datetime.now(WIB)
        
        temp = float(data.get("temperature", 25.0))
        hum = float(data.get("humidity", 65.0))
        soil = float(data.get("soil_moisture", 50.0))
        smoke = float(data.get("smoke_ppm", 90.0))
        dev_id = str(data.get("device_id", "ESP32_FIRE_NODE_01"))

        with data_lock:
            override = latest_state["control_override"]
            if override and override != "AUTO":
                status = override
            else:
                status = evaluate_risk(temp, hum, soil, smoke)
            
            risk_pct = calculate_risk_percentage(temp, hum, soil, smoke)
            
            record = {
                "timestamp": now.strftime("%Y-%m-%d %H:%M:%S"),
                "device_id": dev_id,
                "temperature": round(temp, 1),
                "humidity": round(hum, 1),
                "soil_moisture": round(soil, 1),
                "smoke_ppm": int(smoke),
                "status": status,
                "risk_index": risk_pct
            }
            
            latest_state.update(record)
            latest_state["messages_received"] += 1
            latest_state["last_telemetry_time"] = now.strftime("%H:%M:%S WIB")
            telemetry_history.append(record)
        
        # Kirim sinyal kontrol balik ke Wokwi ESP32
        resp_cmd = json.dumps({"status": status, "risk": status, "source": "Colab_Server"})
        client.publish(TOPIC_CONTROL, resp_cmd)
        
        print(f"[{now.strftime('%H:%M:%S')}] Wokwi Data: Suhu={temp:.1f}°C, Tanah={soil:.1f}%, Asap={smoke:.0f}ppm -> Status Server: {status}")

    except Exception as exc:
        print(f"⚠️ [MQTT] Gagal parse pesan: {exc}")

mqtt_client.on_connect = on_connect
mqtt_client.on_message = on_message
mqtt_client.reconnect_delay_set(min_delay=1, max_delay=30)

try:
    mqtt_client.connect_async(MQTT_BROKER, MQTT_PORT, keepalive=60)
    mqtt_client.loop_start()
    print("Menghubungkan ke broker HiveMQ...")
except Exception as e:
    print(f"Koneksi MQTT error: {e}")


### **4 — REST API Server (FastAPI)**
Menyediakan endpoint REST API untuk melayani data real-time, injeksi payload pengujian, dan menyajikan `dashboard.html`.

In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

nest_asyncio.apply()

api_app = FastAPI(title="Karhutla Vanguard Monitoring API")

api_app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class InjectedTelemetry(BaseModel):
    device_id: Optional[str] = "ESP32_FIRE_NODE_01"
    temperature: float
    humidity: float
    soil_moisture: float
    smoke_ppm: float
    status: Optional[str] = None

class ControlCommand(BaseModel):
    status: Optional[str] = None
    mode: Optional[str] = None

@api_app.get("/", response_class=HTMLResponse)
def serve_dashboard():
    if not os.path.exists(DASHBOARD_FILE):
        raise HTTPException(status_code=404, detail="File dashboard.html tidak ditemukan")
    with open(DASHBOARD_FILE, "r", encoding="utf-8") as f:
        return HTMLResponse(content=f.read())

@api_app.get("/api/telemetry/latest")
def get_latest_telemetry():
    with data_lock:
        return JSONResponse(content=dict(latest_state))

@api_app.get("/api/telemetry/history")
def get_telemetry_history():
    with data_lock:
        return JSONResponse(content=list(telemetry_history))

@api_app.post("/api/telemetry/inject")
def inject_telemetry(payload: InjectedTelemetry):
    now = datetime.now(WIB)
    with data_lock:
        status = payload.status or evaluate_risk(
            payload.temperature, payload.humidity,
            payload.soil_moisture, payload.smoke_ppm
        )
        risk_pct = calculate_risk_percentage(
            payload.temperature, payload.humidity,
            payload.soil_moisture, payload.smoke_ppm
        )
        record = {
            "timestamp": now.strftime("%Y-%m-%d %H:%M:%S"),
            "device_id": payload.device_id,
            "temperature": round(payload.temperature, 1),
            "humidity": round(payload.humidity, 1),
            "soil_moisture": round(payload.soil_moisture, 1),
            "smoke_ppm": int(payload.smoke_ppm),
            "status": status,
            "risk_index": risk_pct
        }
        latest_state.update(record)
        latest_state["messages_received"] += 1
        latest_state["last_telemetry_time"] = now.strftime("%H:%M:%S WIB")
        telemetry_history.append(record)

    # Teruskan perintah ke ESP32 Wokwi melalui MQTT
    mqtt_client.publish(TOPIC_CONTROL, json.dumps({"status": status, "risk": status, "source": "Manual_Inject"}))
    return {"success": True, "data": record}

@api_app.post("/api/control")
def set_control_override(cmd: ControlCommand):
    with data_lock:
        if cmd.mode == "AUTO":
            latest_state["control_override"] = "AUTO"
            status = evaluate_risk(
                latest_state["temperature"], latest_state["humidity"],
                latest_state["soil_moisture"], latest_state["smoke_ppm"]
            )
        else:
            status = cmd.status or "Normal"
            latest_state["control_override"] = status
        latest_state["status"] = status

    # Publish override ke ESP32 Wokwi
    mqtt_client.publish(TOPIC_CONTROL, json.dumps({"status": status, "risk": status, "source": "Web_Override"}))
    return {"success": True, "status": status}

@api_app.get("/api/health")
def health_check():
    return {"status": "ok", "mqtt": latest_state["mqtt_connected"], "received": latest_state["messages_received"]}

print("✅ FastAPI Application berhasil dikonfigurasi!")


### **5 — Jalankan Server & Buka Dashboard di Google Colab**
Menjalankan server web Uvicorn di thread background dan membuka Dashboard secara interaktif di Google Colab via `output.serve_kernel_port_as_iframe`.

In [ ]:
import requests

def run_uvicorn():
    config = uvicorn.Config(api_app, host="0.0.0.0", port=HTTP_PORT, log_level="warning")
    server = uvicorn.Server(config)
    server.run()

# Jalankan server di background daemon thread
server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()

# Tunggu hingga server siap melayani request
server_ready = False
for _ in range(20):
    try:
        resp = requests.get(f"http://127.0.0.1:{HTTP_PORT}/api/health", timeout=1)
        if resp.status_code == 200:
            server_ready = True
            break
    except:
        time.sleep(0.5)

if server_ready:
    print(f"🚀 REST API & Dashboard Server aktif di port {HTTP_PORT}!")
    try:
        from google.colab import output
        print("🌟 Menampilkan Dashboard di dalam antarmuka Google Colab...")
        output.serve_kernel_port_as_iframe(HTTP_PORT, height=850)
        output.serve_kernel_port_as_window(HTTP_PORT)
    except ImportError:
        print(f"💻 Berjalan di mesin lokal. Buka di browser Anda: http://localhost:{HTTP_PORT}")
else:
    print("❌ Server gagal memulai. Periksa log kesalahan di atas.")


### **6 — Live Console Telemetry Stream**
Cell opsional untuk memonitor paket data telemetri yang masuk dari Wokwi secara realtime di konsol Colab.

In [ ]:
print("📡 Menunggu telemetri dari ESP32 Wokwi (Tekan tombol Stop / Interupt pada cell untuk berhenti)...")
last_count = -1

try:
    while True:
        with data_lock:
            cur = dict(latest_state)
        
        if cur["messages_received"] != last_count:
            last_count = cur["messages_received"]
            print(
                f"[{cur['last_telemetry_time'] or '--'}] "
                f"Dev: {cur['device_id']} | "
                f"Suhu: {cur['temperature']}°C | "
                f"Tanah: {cur['soil_moisture']}% | "
                f"Asap: {cur['smoke_ppm']} ppm | "
                f"Risiko: {cur['risk_index']}% -> [{cur['status']}]"
            )
        time.sleep(1)
except KeyboardInterrupt:
    print("\n⏹️ Monitoring konsol dihentikan.")
